# Agent 3 – Détection de Research Gaps

## Rôle et Responsabilités

Cet agent est le **cœur analytique** du système. Il identifie les **lacunes de recherche** (research gaps) en analysant la synthèse produite par l'Agent 2. Il utilise une technique avancée de **Chain-of-Thought (CoT)** pour structurer son raisonnement avant de formuler ses conclusions.

### Types de gaps identifiés

| Type | Description |
|------|-------------|
| **Methodological Gap** | Méthodes qui n'ont pas encore été appliquées au domaine |
| **Domain Gap** | Sous-domaines ou sujets non couverts par la littérature |
| **Temporal Gap** | Approches anciennes qui n'ont pas été mises à jour |
| **Application Gap** | Théories bien établies non appliquées en pratique |

### Technique Chain-of-Thought (CoT)

L'agent demande à Claude de :
1. **Raisonner étape par étape** dans des balises `<analysis>` (raisonnement interne)
2. **Formuler ses conclusions** en JSON structuré dans des balises `<gaps>` (sortie finale)

Cette approche améliore la qualité et la fiabilité des gaps détectés.

### Modèle utilisé
`claude-sonnet-4-5` – pour un raisonnement académique de haut niveau

In [1]:
# ============================================================
# Installation des dépendances
# ============================================================
# !pip install anthropic python-dotenv

# ============================================================
# Imports
# ============================================================
import llm_adapter as anthropic  # adaptateur Gemini (gratuit)
import json
import re
import os
import time
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

print("Imports réussis.")
print(f"LLM adapter version : {anthropic.__version__}")

Imports réussis.
LLM adapter version : llm_adapter/gemini-2.0-flash


In [2]:
# ============================================================
# Chargement de la synthèse et initialisation du client
# ============================================================

# Charger les variables d'environnement
load_dotenv(dotenv_path=Path("../.env"))
ANTHROPIC_API_KEY = os.getenv("GEMINI_API_KEY", "")

if not ANTHROPIC_API_KEY:
    print("[ATTENTION] ANTHROPIC_API_KEY non trouvée dans .env")
else:
    print(f"Clé API Anthropic chargée : {ANTHROPIC_API_KEY[:8]}...")

# Constantes
LOGS_DIR       = "../logs"
RAPPORT_DIR    = "../logs"
GAP_MODEL      = "claude-sonnet-4-5"
MAX_TOKENS     = 4096

# Initialiser le client Anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f"Client Anthropic initialisé (modèle : {GAP_MODEL})")

# Charger la synthèse produite par l'Agent 2
synthesis_path = Path(LOGS_DIR) / "synthesis_result.json"

if synthesis_path.exists():
    with open(synthesis_path, "r", encoding="utf-8") as f:
        synthesis_data = json.load(f)
    print(f"Synthèse chargée depuis : {synthesis_path}")
    print(f"Thèmes analysés : {len(synthesis_data.get('themes_analyzed', []))}")
    print(f"Synthèses réussies : {synthesis_data.get('usage_stats', {}).get('successful_syntheses', 0)}")
else:
    print(f"[ATTENTION] Fichier de synthèse non trouvé : {synthesis_path}")
    print("Exécutez d'abord l'Agent 2 (02_agent_synthese.ipynb)")
    # Créer une synthèse minimale pour permettre le test
    synthesis_data = {
        "status": "no_file",
        "themes_analyzed": [],
        "theme_syntheses": [],
        "aggregated_findings": [],
        "aggregated_methodologies": []
    }

Clé API Anthropic chargée : AIza****...


Client Anthropic initialisé (modèle : claude-sonnet-4-5)
Synthèse chargée depuis : ..\logs\synthesis_result.json
Thèmes analysés : 5
Synthèses réussies : 0


In [3]:
# ============================================================
# Prompt système pour la détection de gaps
# ============================================================

GAP_DETECTION_SYSTEM_PROMPT = """You are an expert academic researcher with 20+ years of experience in systematic literature reviews and research strategy. Your specialty is identifying research gaps — the unexplored spaces where new scientific contributions can have the highest impact.

Your task is to analyze a synthesis of a research domain and systematically identify research gaps. You will reason step-by-step before providing your structured output.

## Gap Categories to Analyze

1. **Methodological Gaps**: Research methods or techniques that have NOT been applied to this domain yet, despite being available and potentially useful. Ask: "What approaches from adjacent fields could be imported here?"

2. **Domain Gaps**: Specific sub-topics, populations, contexts, or aspects of the domain that are underrepresented or completely absent from the literature. Ask: "What aspects of this problem remain unexplored?"

3. **Temporal Gaps**: Older approaches or findings that have not been updated with modern techniques (e.g., pre-deep-learning methods that should be revisited with neural approaches). Ask: "What older work needs to be updated?"

4. **Application Gaps**: Theoretical findings or proven techniques that have not been implemented or validated in real-world practice. Ask: "What works in theory but hasn't been tested in practice?"

## Output Format

First, reason through your analysis step-by-step inside <analysis> tags.
Then provide your final structured answer inside <gaps> tags as a JSON object.

The JSON must follow this exact structure:
{
  "research_gaps": [
    {
      "gap_id": "GAP-001",
      "gap_type": "Methodological|Domain|Temporal|Application",
      "title": "Concise title of the gap",
      "description": "Detailed description of what is missing",
      "evidence": "Evidence from the synthesis that reveals this gap",
      "justification": "Why this gap matters and how it was identified",
      "suggested_research_direction": "Specific, actionable research direction to address this gap",
      "estimated_impact": "High|Medium|Low",
      "feasibility": "High|Medium|Low",
      "related_themes": ["theme1", "theme2"]
    }
  ],
  "priority_ranking": ["GAP-001", "GAP-002"],
  "cross_cutting_observations": "Observations that span multiple gaps",
  "recommended_next_steps": "Overall strategic recommendations for the research community"
}

## Quality Standards
- Each gap must be specific and actionable (not vague like "more research is needed")
- Each gap must be grounded in evidence from the provided synthesis
- Aim for 5-10 high-quality gaps rather than many superficial ones
- Priority ranking should reflect Impact × Feasibility
- Be constructive: each gap should include a concrete research direction"""

print("Prompt GAP_DETECTION_SYSTEM_PROMPT défini.")
print(f"Longueur : {len(GAP_DETECTION_SYSTEM_PROMPT)} caractères")

Prompt GAP_DETECTION_SYSTEM_PROMPT défini.
Longueur : 2730 caractères


In [4]:
# ============================================================
# Détection de gaps avec Chain-of-Thought
# ============================================================

def detect_gaps_with_cot(synthesis_text: str, client: anthropic.Anthropic) -> dict:
    """Détecte les research gaps avec raisonnement Chain-of-Thought.
    
    Demande à Claude de :
    1. Raisonner étape par étape dans <analysis> (CoT interne)
    2. Fournir la réponse structurée en JSON dans <gaps>
    
    Args:
        synthesis_text: Texte de la synthèse globale de la littérature.
        client: Instance du client Anthropic.
        
    Returns:
        Dictionnaire contenant :
          - reasoning (str): Raisonnement CoT extrait des balises <analysis>
          - gaps (dict): Gaps structurés extraits des balises <gaps>
          - status (str): 'success', 'partial' ou 'error'
          - metadata (dict): Informations sur l'exécution (tokens, durée)
    """
    print("Détection des research gaps avec Chain-of-Thought...")
    print(f"Longueur de la synthèse : {len(synthesis_text)} caractères")

    # Construire le prompt utilisateur avec instruction CoT explicite
    user_message = f"""Please analyze the following research domain synthesis and identify research gaps.

IMPORTANT: Follow the two-step process:
1. First, reason step-by-step inside <analysis> tags. Consider each gap category systematically.
2. Then, provide your structured JSON answer inside <gaps> tags.

## Research Domain Synthesis

{synthesis_text}

Begin your step-by-step analysis now:"""

    start_time = time.time()

    try:
        # Appel à l'API Claude avec max_tokens élevé pour le CoT
        print(f"Appel à {GAP_MODEL} (max_tokens={MAX_TOKENS})...")

        response = client.messages.create(
            model=GAP_MODEL,
            max_tokens=MAX_TOKENS,
            system=GAP_DETECTION_SYSTEM_PROMPT,
            messages=[
                {"role": "user", "content": user_message}
            ]
        )

        raw_response = response.content[0].text
        duration = round(time.time() - start_time, 2)
        print(f"Réponse reçue en {duration}s | {response.usage.input_tokens} tokens in, {response.usage.output_tokens} tokens out")

        # Extraire le raisonnement CoT (balises <analysis>)
        analysis_match = re.search(r"<analysis>(.*?)</analysis>", raw_response, re.DOTALL)
        reasoning = analysis_match.group(1).strip() if analysis_match else ""

        if not reasoning:
            # Tentative alternative : texte avant les balises <gaps>
            gaps_start = raw_response.find("<gaps>")
            if gaps_start > 0:
                reasoning = raw_response[:gaps_start].strip()

        # Extraire le JSON structuré (balises <gaps>)
        gaps_match = re.search(r"<gaps>(.*?)</gaps>", raw_response, re.DOTALL)

        if gaps_match:
            gaps_json_text = gaps_match.group(1).strip()
            # Nettoyer les balises markdown éventuelles
            if gaps_json_text.startswith("```json"):
                gaps_json_text = gaps_json_text[7:]
            if gaps_json_text.startswith("```"):
                gaps_json_text = gaps_json_text[3:]
            if gaps_json_text.endswith("```"):
                gaps_json_text = gaps_json_text[:-3]

            gaps_data = json.loads(gaps_json_text.strip())
            status = "success"
        else:
            # Fallback : chercher le JSON directement dans la réponse
            print("  [ATTENTION] Balises <gaps> non trouvées, recherche directe du JSON...")
            json_match = re.search(r"\{[\s\S]*\"research_gaps\"[\s\S]*\}", raw_response)
            if json_match:
                gaps_data = json.loads(json_match.group(0))
                status = "partial"
            else:
                raise ValueError("Aucun JSON de gaps trouvé dans la réponse")

        return {
            "reasoning": reasoning,
            "gaps": gaps_data,
            "status": status,
            "metadata": {
                "model": GAP_MODEL,
                "input_tokens": response.usage.input_tokens,
                "output_tokens": response.usage.output_tokens,
                "duration_seconds": duration,
                "n_gaps_detected": len(gaps_data.get("research_gaps", []))
            }
        }

    except json.JSONDecodeError as je:
        print(f"  [ERREUR JSON] Parsing échoué : {je}")
        return {
            "reasoning": reasoning if 'reasoning' in locals() else "",
            "gaps": {
                "research_gaps": [],
                "priority_ranking": [],
                "cross_cutting_observations": "Erreur de parsing JSON",
                "recommended_next_steps": "Relancer l'analyse"
            },
            "status": "json_error",
            "metadata": {"error": str(je)}
        }

    except anthropic.APIError as ae:
        print(f"  [ERREUR API] {ae}")
        return {
            "reasoning": "",
            "gaps": {
                "research_gaps": [],
                "priority_ranking": [],
                "cross_cutting_observations": f"Erreur API : {str(ae)}",
                "recommended_next_steps": "Vérifier la clé API et le quota"
            },
            "status": "api_error",
            "metadata": {"error": str(ae)}
        }

    except Exception as e:
        print(f"  [ERREUR] {e}")
        return {
            "reasoning": "",
            "gaps": {
                "research_gaps": [],
                "priority_ranking": [],
                "cross_cutting_observations": f"Erreur : {str(e)}",
                "recommended_next_steps": "Vérifier les logs"
            },
            "status": "error",
            "metadata": {"error": str(e)}
        }


print("Fonction detect_gaps_with_cot définie.")

Fonction detect_gaps_with_cot définie.


In [5]:
# ============================================================
# Génération du rapport de gaps en Markdown
# ============================================================

def generate_gap_report(gaps_result: dict) -> str:
    """Génère un rapport Markdown lisible à partir des gaps détectés.
    
    Formate les gaps en sections claires avec niveaux de priorité,
    descriptions détaillées et recommandations de recherche.
    
    Args:
        gaps_result: Dictionnaire retourné par detect_gaps_with_cot().
        
    Returns:
        Chaîne Markdown formatée prête à être sauvegardée.
    """
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    gaps_data = gaps_result.get("gaps", {})
    research_gaps = gaps_data.get("research_gaps", [])
    metadata = gaps_result.get("metadata", {})

    # En-tête du rapport
    lines = [
        "# Rapport de Détection des Research Gaps",
        "",
        f"**Date de génération** : {now}  ",
        f"**Modèle utilisé** : {metadata.get('model', 'N/A')}  ",
        f"**Statut** : {gaps_result.get('status', 'N/A')}  ",
        f"**Nombre de gaps identifiés** : {len(research_gaps)}  ",
        "",
        "---",
        ""
    ]

    # Résumé exécutif
    lines += [
        "## Résumé Exécutif",
        "",
        gaps_data.get("cross_cutting_observations", "Aucune observation transversale disponible."),
        "",
        "### Recommandations Stratégiques",
        "",
        gaps_data.get("recommended_next_steps", "Aucune recommandation disponible."),
        "",
        "---",
        ""
    ]

    # Ordre de priorité
    priority_ranking = gaps_data.get("priority_ranking", [])
    if priority_ranking:
        lines += [
            "## Classement par Priorité",
            ""
        ]
        for rank, gap_id in enumerate(priority_ranking, 1):
            # Trouver le gap correspondant
            gap = next((g for g in research_gaps if g.get("gap_id") == gap_id), None)
            if gap:
                impact = gap.get("estimated_impact", "N/A")
                feasibility = gap.get("feasibility", "N/A")
                lines.append(f"{rank}. **{gap_id}** – {gap.get('title', 'N/A')} (Impact: {impact} | Faisabilité: {feasibility})")
        lines += ["", "---", ""]

    # Gaps par catégorie
    gap_types = ["Methodological", "Domain", "Temporal", "Application"]
    type_labels = {
        "Methodological": "Gaps Méthodologiques",
        "Domain": "Gaps de Domaine",
        "Temporal": "Gaps Temporels",
        "Application": "Gaps d'Application"
    }

    for gap_type in gap_types:
        type_gaps = [g for g in research_gaps if g.get("gap_type") == gap_type]
        if not type_gaps:
            continue

        lines += [
            f"## {type_labels.get(gap_type, gap_type)}",
            ""
        ]

        for gap in type_gaps:
            gap_id = gap.get("gap_id", "N/A")
            impact = gap.get("estimated_impact", "N/A")
            feasibility = gap.get("feasibility", "N/A")

            # Badge de priorité
            impact_badge = {"High": "HAUTE", "Medium": "MOYENNE", "Low": "BASSE"}.get(impact, impact)
            feasibility_badge = {"High": "HAUTE", "Medium": "MOYENNE", "Low": "BASSE"}.get(feasibility, feasibility)

            lines += [
                f"### {gap_id} – {gap.get('title', 'N/A')}",
                "",
                f"**Impact estimé** : {impact_badge} | **Faisabilité** : {feasibility_badge}",
                "",
                "**Description** :",
                "",
                gap.get("description", "N/A"),
                "",
                "**Evidence de la synthèse** :",
                "",
                f"_{gap.get('evidence', 'N/A')}_",
                "",
                "**Justification** :",
                "",
                gap.get("justification", "N/A"),
                "",
                "**Direction de recherche suggérée** :",
                "",
                f"> {gap.get('suggested_research_direction', 'N/A')}",
                "",
                f"**Thèmes liés** : {', '.join(gap.get('related_themes', []))}",
                "",
                "---",
                ""
            ]

    # Section CoT (raisonnement)
    reasoning = gaps_result.get("reasoning", "")
    if reasoning:
        lines += [
            "## Raisonnement Chain-of-Thought (Extrait)",
            "",
            "<details>",
            "<summary>Cliquer pour afficher le raisonnement interne de Claude</summary>",
            "",
            "```",
            reasoning[:2000] + ("..." if len(reasoning) > 2000 else ""),
            "```",
            "",
            "</details>",
            ""
        ]

    # Pied de page
    lines += [
        "---",
        "",
        f"*Rapport généré automatiquement par l'Agent 3 du SMA – {now}*"
    ]

    return "\n".join(lines)


print("Fonction generate_gap_report définie.")

Fonction generate_gap_report définie.


In [6]:
# ============================================================
# Agent principal de Détection de Gaps
# ============================================================

def run_gap_detection_agent() -> dict:
    """Exécute l'agent complet de détection de research gaps.
    
    Orchestre :
    1. Chargement et préparation de la synthèse
    2. Détection des gaps avec Chain-of-Thought
    3. Génération du rapport Markdown
    4. Sauvegarde des résultats JSON et du rapport
    5. Affichage du résumé
    
    Returns:
        Dictionnaire complet avec gaps, rapport et métadonnées.
    """
    start_time = time.time()
    print("=" * 60)
    print("AGENT 3 – DÉTECTION DE RESEARCH GAPS")
    print("=" * 60)
    print(f"Démarrage : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()

    # Étape 1 : Charger et préparer la synthèse
    synthesis_path = Path(LOGS_DIR) / "synthesis_result.json"

    if not synthesis_path.exists():
        print(f"[ERREUR] Fichier de synthèse non trouvé : {synthesis_path}")
        return {"status": "error", "error": "Fichier synthesis_result.json manquant"}

    with open(synthesis_path, "r", encoding="utf-8") as f:
        synthesis_data = json.load(f)

    # Construire un texte de synthèse condensé pour l'analyse
    synthesis_parts = []

    # En-tête
    themes_analyzed = synthesis_data.get("themes_analyzed", [])
    synthesis_parts.append(f"## Research Domain Overview\nThemes analyzed: {', '.join(themes_analyzed)}\n")

    # Synthèses par thème
    for synth in synthesis_data.get("theme_syntheses", []):
        if synth.get("status") != "success":
            continue

        theme = synth.get("theme", "")
        synthesis_parts.append(f"\n### Theme: {theme}")

        # Sous-thèmes
        themes = synth.get("themes", [])
        if themes:
            synthesis_parts.append("\n**Sub-themes identified:**")
            for t in themes:
                synthesis_parts.append(f"- {t.get('name', '')}: {t.get('description', '')} (Frequency: {t.get('frequency', 'N/A')})")

        # Méthodologies
        methodologies = synth.get("key_methodologies", [])
        if methodologies:
            synthesis_parts.append("\n**Key methodologies:**")
            for m in methodologies:
                synthesis_parts.append(f"- {m.get('method', '')}: {m.get('description', '')}")

        # Résultats principaux
        findings = synth.get("main_findings", [])
        if findings:
            synthesis_parts.append("\n**Main findings:**")
            for f in findings:
                synthesis_parts.append(f"- {f.get('finding', '')}")

        # Évolution temporelle
        temporal = synth.get("temporal_evolution", {})
        if temporal:
            synthesis_parts.append(f"\n**Temporal evolution:** {temporal.get('trend', 'N/A')}")

    synthesis_text = "\n".join(synthesis_parts)

    print(f"Synthèse préparée : {len(synthesis_text)} caractères")
    print(f"Thèmes inclus : {len([s for s in synthesis_data.get('theme_syntheses', []) if s.get('status') == 'success'])}")

    # Fallback si la synthèse est vide
    if len(synthesis_text.strip()) < 100:
        synthesis_text = "No detailed synthesis available. Please analyze the domain of automated literature review and research gap detection in general terms based on your knowledge."
        print("[ATTENTION] Synthèse très courte, utilisation du fallback générique")

    # Étape 2 : Détecter les gaps avec CoT
    print()
    gaps_result = detect_gaps_with_cot(synthesis_text, client)

    # Étape 3 : Générer le rapport Markdown
    print("\nGénération du rapport Markdown...")
    gap_report_md = generate_gap_report(gaps_result)

    # Étape 4 : Sauvegarder les résultats
    logs_path = Path(LOGS_DIR)
    logs_path.mkdir(parents=True, exist_ok=True)

    # Sauvegarder le JSON
    gaps_json_path = logs_path / "gaps_result.json"
    full_result = {
        "status": gaps_result.get("status"),
        "gaps": gaps_result.get("gaps", {}),
        "reasoning_excerpt": gaps_result.get("reasoning", "")[:1000],
        "metadata": gaps_result.get("metadata", {}),
        "generated_at": datetime.now().isoformat()
    }

    with open(gaps_json_path, "w", encoding="utf-8") as f:
        json.dump(full_result, f, indent=2, ensure_ascii=False)

    # Sauvegarder le rapport Markdown
    report_md_path = logs_path / "gap_report.md"
    with open(report_md_path, "w", encoding="utf-8") as f:
        f.write(gap_report_md)

    print(f"Résultats JSON sauvegardés : {gaps_json_path}")
    print(f"Rapport Markdown sauvegardé : {report_md_path}")

    # Étape 5 : Afficher le résumé
    research_gaps = gaps_result.get("gaps", {}).get("research_gaps", [])
    duration = round(time.time() - start_time, 2)

    print()
    print("=" * 60)
    print("RÉSUMÉ DE LA DÉTECTION DE GAPS")
    print("=" * 60)
    print(f"Statut               : {gaps_result.get('status', 'N/A')}")
    print(f"Gaps identifiés      : {len(research_gaps)}")
    print(f"Durée totale         : {duration}s")
    print()

    if research_gaps:
        print(f"{'ID':<10} {'Type':<18} {'Impact':<8} {'Faisab.':<8} Titre")
        print("-" * 60)
        for gap in research_gaps:
            gap_id = gap.get("gap_id", "N/A")
            gap_type = gap.get("gap_type", "N/A")[:16]
            impact = gap.get("estimated_impact", "N/A")
            feasibility = gap.get("feasibility", "N/A")
            title = gap.get("title", "N/A")[:30]
            print(f"{gap_id:<10} {gap_type:<18} {impact:<8} {feasibility:<8} {title}")

    full_result["report_markdown"] = gap_report_md
    return full_result


print("Fonction run_gap_detection_agent définie.")

Fonction run_gap_detection_agent définie.


In [7]:
# ============================================================
# Exécution de l'Agent de Détection de Gaps
# ============================================================

# Lancer l'agent
gaps_result = run_gap_detection_agent()

# Afficher un aperçu du rapport
print()
print("=" * 60)
print("APERÇU DU RAPPORT DE GAPS")
print("=" * 60)

report_preview = gaps_result.get("report_markdown", "")
if report_preview:
    # Afficher les 50 premières lignes du rapport
    preview_lines = report_preview.split("\n")[:50]
    print("\n".join(preview_lines))
    if len(report_preview.split("\n")) > 50:
        print(f"\n... (rapport tronqué, {len(report_preview.split(chr(10)))} lignes au total)")
    print()
    print("Rapport complet disponible dans : ../logs/gap_report.md")
else:
    print("[ATTENTION] Rapport vide ou erreur lors de la génération")

AGENT 3 – DÉTECTION DE RESEARCH GAPS
Démarrage : 2026-05-15 12:16:12

Synthèse préparée : 198 caractères
Thèmes inclus : 0

Détection des research gaps avec Chain-of-Thought...
Longueur de la synthèse : 198 caractères
Appel à claude-sonnet-4-5 (max_tokens=4096)...


  [ERREUR] 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 46.621462225s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog

## Analyse de Coût

Cette section estime le coût des appels API Anthropic pour l'Agent 3.

### Tarification Claude Sonnet (claude-sonnet-4-5)
- **Input tokens** : $3 / million de tokens
- **Output tokens** : $15 / million de tokens

### Formule de calcul
```
Coût = (input_tokens / 1_000_000) × $3 + (output_tokens / 1_000_000) × $15
```

In [8]:
# ============================================================
# Estimation du coût des appels API
# ============================================================

# Tarification Claude claude-sonnet-4-5 (USD par million de tokens)
SONNET_INPUT_COST_PER_MILLION  = 3.0   # $3 / million tokens input
SONNET_OUTPUT_COST_PER_MILLION = 15.0  # $15 / million tokens output

# Récupérer les métriques d'usage de l'agent 3
agent3_metadata = gaps_result.get("metadata", {})
agent3_input_tokens  = agent3_metadata.get("input_tokens", 0)
agent3_output_tokens = agent3_metadata.get("output_tokens", 0)

# Récupérer les métriques de l'agent 2 si disponibles
agent2_input_tokens  = 0
agent2_output_tokens = 0
synthesis_path = Path(LOGS_DIR) / "synthesis_result.json"
if synthesis_path.exists():
    with open(synthesis_path, "r", encoding="utf-8") as f:
        synth_data = json.load(f)
    usage = synth_data.get("usage_stats", {})
    agent2_input_tokens  = usage.get("total_input_tokens", 0)
    agent2_output_tokens = usage.get("total_output_tokens", 0)

# Calcul des coûts
def calculate_cost(input_tokens: int, output_tokens: int) -> dict:
    """Calcule le coût estimé pour un appel API Claude Sonnet."""
    input_cost  = (input_tokens  / 1_000_000) * SONNET_INPUT_COST_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * SONNET_OUTPUT_COST_PER_MILLION
    total_cost  = input_cost + output_cost
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "input_cost_usd": round(input_cost, 6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd": round(total_cost, 6)
    }

agent2_cost = calculate_cost(agent2_input_tokens, agent2_output_tokens)
agent3_cost = calculate_cost(agent3_input_tokens, agent3_output_tokens)

total_input  = agent2_input_tokens  + agent3_input_tokens
total_output = agent2_output_tokens + agent3_output_tokens
total_cost   = calculate_cost(total_input, total_output)

# Affichage du rapport de coût
print("=" * 60)
print("RAPPORT DE COÛT API ANTHROPIC")
print("=" * 60)
print(f"Modèle : {GAP_MODEL}")
print(f"Tarif  : Input ${SONNET_INPUT_COST_PER_MILLION}/M tokens | Output ${SONNET_OUTPUT_COST_PER_MILLION}/M tokens")
print()
print(f"{'Agent':<25} {'Tokens In':>12} {'Tokens Out':>12} {'Coût USD':>12}")
print("-" * 65)
print(f"{'Agent 2 (Synthèse)':<25} {agent2_cost['input_tokens']:>12,} {agent2_cost['output_tokens']:>12,} {'$' + str(agent2_cost['total_cost_usd']):>12}")
print(f"{'Agent 3 (Gaps)':<25} {agent3_cost['input_tokens']:>12,} {agent3_cost['output_tokens']:>12,} {'$' + str(agent3_cost['total_cost_usd']):>12}")
print("-" * 65)
print(f"{'TOTAL SMA':<25} {total_cost['input_tokens']:>12,} {total_cost['output_tokens']:>12,} {'$' + str(total_cost['total_cost_usd']):>12}")
print()

# Projections de coût
print("Projections de coût :")
print(f"  Pour 10 exécutions   : ${round(total_cost['total_cost_usd'] * 10, 4)} USD")
print(f"  Pour 100 exécutions  : ${round(total_cost['total_cost_usd'] * 100, 4)} USD")
print(f"  Pour 1000 exécutions : ${round(total_cost['total_cost_usd'] * 1000, 4)} USD")
print()
print("Note : L'Agent 1 (Ingestion) utilise sentence-transformers en local (coût = $0)")
print("       Les embeddings ChromaDB sont également générés en local.")

RAPPORT DE COÛT API ANTHROPIC
Modèle : claude-sonnet-4-5
Tarif  : Input $3.0/M tokens | Output $15.0/M tokens

Agent                        Tokens In   Tokens Out     Coût USD
-----------------------------------------------------------------
Agent 2 (Synthèse)                   0            0         $0.0
Agent 3 (Gaps)                       0            0         $0.0
-----------------------------------------------------------------
TOTAL SMA                            0            0         $0.0

Projections de coût :
  Pour 10 exécutions   : $0.0 USD
  Pour 100 exécutions  : $0.0 USD
  Pour 1000 exécutions : $0.0 USD

Note : L'Agent 1 (Ingestion) utilise sentence-transformers en local (coût = $0)
       Les embeddings ChromaDB sont également générés en local.
